# Foodwise: train and evaluate the ranking model

Run this notebook in Google Colab. It queries USDA FoodData Central's public search API for real nutrition records and trains a linear model. USDA provides food nutrient data, **not** Thai meal prices, preferences, or ratings. Labels here are transparent weak supervision: a proxy utility favors protein density and lower sodium, with randomized UI preference/budget signals. Metrics measure agreement with this proxy only, never user satisfaction or medical outcomes. The app's small Thai menu/price catalog is a separate approximate demo source.

Source and API docs: https://fdc.nal.usda.gov/api-guide/ and https://fdc.nal.usda.gov/download-datasets/ . Get a free key at https://fdc.nal.usda.gov/api-key-signup.html .

In [ ]:
%pip -q install requests pandas scikit-learn
import os, getpass, requests, numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
try:
    from google.colab import userdata
    API_KEY = userdata.get('FDC_API_KEY')
except Exception:
    API_KEY = None
if not API_KEY: API_KEY = getpass.getpass('USDA FDC API key: ')
assert API_KEY, 'A FoodData Central API key is required.'

## Fetch real records
Search the public FDC API across several common foods. Keep FDC IDs and descriptions as provenance. Nutrients are normalized per 100g when supplied by FDC.

In [ ]:
terms = ['chicken breast','tofu','brown rice','egg','salmon','oats','banana','lentils','spinach','yogurt','shrimp','potato','beans','pork loin','whole wheat bread']
rows=[]
for term in terms:
    response=requests.post('https://api.nal.usda.gov/fdc/v1/foods/search',params={'api_key':API_KEY},json={'query':term,'dataType':['Foundation','SR Legacy'],'pageSize':25},timeout=30)
    response.raise_for_status()
    for food in response.json().get('foods',[]):
        nutrients={n['nutrientId']:n.get('value') for n in food.get('foodNutrients',[]) if n.get('nutrientId') in [1008,1003,1005,1004,1093]}
        rows.append({'fdc_id':food.get('fdcId'),'description':food.get('description',''),'data_type':food.get('dataType',''),'kcal':nutrients.get(1008),'protein_g':nutrients.get(1003),'carbs_g':nutrients.get(1005),'fat_g':nutrients.get(1004),'sodium_mg':nutrients.get(1093)})
df=pd.DataFrame(rows).drop_duplicates('fdc_id').dropna(subset=['kcal','protein_g','sodium_mg'])
df=df[(df.kcal>0)&(df.protein_g>=0)&(df.sodium_mg>=0)].reset_index(drop=True)
assert len(df)>=30, f'Only {len(df)} valid records; check the key or rerun.'
df.to_csv('/content/usda_fooddata_central_sample.csv',index=False)
print(f'{len(df)} unique FDC foods. CSV saved in the Colab file pane as usda_fooddata_central_sample.csv')
df.head()

## Build features and weak labels
The six feature names match the Go inference code. Nutrition features come from USDA observations. Since USDA contains no price or user preference signal, those two request-context features are sampled for this proxy experiment. The proxy label is deliberately explicit; replace it with consented user feedback before claiming preference learning.

In [ ]:
rng=np.random.default_rng(42)
budget_fit=rng.uniform(.25,1,len(df))
preference_match=rng.uniform(0,1,len(df))
protein=np.clip(df.protein_g.to_numpy()/35,0,1)
low_sodium=np.clip(1-df.sodium_mg.to_numpy()/1800,0,1)
calorie_fit=np.clip(1-np.abs(df.kcal.to_numpy()-500)/500,0,1)
popularity=np.full(len(df),.5)  # No popularity observations in FDC.
feature_names=['budget_fit','protein','low_sodium','preference_match','calorie_fit','popularity']
X=np.column_stack([budget_fit,protein,low_sodium,preference_match,calorie_fit,popularity])
# Weak-label utility: transparent prototype target, not human ratings.
y=.15*budget_fit + .35*protein + .25*low_sodium + .15*preference_match + .10*calorie_fit
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,random_state=42)
model=Ridge(alpha=1e-3).fit(X_train,y_train)
pred=model.predict(X_test)
metrics={'mae':mean_absolute_error(y_test,pred),'rmse':mean_squared_error(y_test,pred)**.5,'r2':r2_score(y_test,pred)}
print('Held-out weak-label proxy metrics:',metrics)
print('These do not measure real user preference, recommendation click-through, or health outcomes.')

## Export weights for Go
The feature order is consumed by `internal/recommender/model.json`. Copying this file replaces the clearly marked baseline weights. `trained: true` means fitted on USDA observations against the proxy label described above.

In [ ]:
import json, pathlib
artifact={'version':'1.0-usda-proxy','features':feature_names,'weights':[float(v) for v in model.coef_],'bias':float(model.intercept_),'trained':True,'note':'Weakly supervised proxy trained on USDA FDC nutrients; not user ratings.'}
out=pathlib.Path('/content/model.json')
out.write_text(json.dumps(artifact,ensure_ascii=False,indent=2),encoding='utf-8')
print('Downloaded model.json and replace internal/recommender/model.json in your project.')
print(json.dumps(artifact,indent=2))
from google.colab import files
files.download(str(out))

## Interpretation and next steps
The fitted model approximates the declared proxy target. It cannot learn whether diners like a dish from this dataset. To move beyond a demonstration, collect consented impressions, saves, and ratings with clear privacy terms; verify Thai recipe nutrition and vendor prices; use a user/item ranking objective; evaluate on a time-separated user holdout with NDCG@K / MAP@K and calibration checks. Never treat a sodium preference filter or this prototype score as clinical advice.